# 04 - Extract Eurostat COMEXT Trade Data

## Purpose
Pull EU import volumes and values for all CBAM-covered products by partner
country from the Eurostat COMEXT database via the SDMX API. This gives us
the trade flow layer: who is exporting how much of what to the EU, and at
what value.

## Dataset
DS-045409: EU trade since 1988 by HS2-4-6 and CN8

Note: DS-059322 (previously referenced in project planning) is no longer
available for dissemination via the API as of May 2026. DS-045409 is the
current equivalent, confirmed via `eurostat.get_toc_df(agency='COMEXT')`.

## API Access
Accessed via the `eurostat` Python package (pip install eurostat).
Base endpoint: `https://ec.europa.eu/eurostat/api/comext/dissemination`

## Dimension Filters
- FREQ: A (annual)
- REPORTER: EU27_2020 (EU aggregate)
- PARTNER: all (open, returns all partner countries)
- PRODUCT: CBAM-covered CN codes (batched by material group)
- FLOW: 1 (imports only)
- PERIOD: 2020-2024
- INDICATORS: VALUE_IN_EUROS and QUANTITY_IN_100KG

## Output
`data/processed/eu_import_trade_flows.csv`

## Notes
- API requires filtering due to dataset size. Full download not permitted.
- Requests batched by material group to avoid exceeding the 5M cell limit.
- QUANTITY_IN_100KG converted to tonnes (divide by 10) in output.
- Trade volumes are EU27 aggregate, not individual member state level.
  Member state breakdown available in DS-045409 if needed later.
- Electricity excluded from scope (cross-border electricity trading
  is a separate CBAM mechanism not relevant to this analysis).


In [ ]:
import pandas as pd
import eurostat
import time
from pathlib import Path

# Paths
defaults_path = Path("/Users/milcahmaryJoseph/Documents/GitHub/cbam-analysis/data/processed/cbam_defaults.csv")
output_path = Path("/Users/milcahmaryJoseph/Documents/GitHub/cbam-analysis/data/processed/eu_import_trade_flows.csv")

# Load CBAM defaults to get CN code list
df_defaults = pd.read_csv(defaults_path, dtype=str)
df_defaults["cn_clean"] = df_defaults["cn_code"].str.replace(" ", "").str.strip()

print(f"Loaded {len(df_defaults)} rows from cbam_defaults.csv")
print(f"Unique CN codes: {df_defaults['cn_clean'].nunique()}")

Loaded 10671 rows from cbam_defaults.csv
Unique CN codes: 262


## Section 1: CN Code Preparation

Assign each CN code to a CBAM material group based on its numeric prefix.
Electricity (27xx) is excluded from the API queries as cross-border
electricity trading is outside this project's analytical scope.
Hydrogen (2804) is separated from other 28xx fertilizer codes.

In [24]:
def assign_material(cn_code):
    cn = str(cn_code).replace(" ", "").strip()
    if cn.startswith("2804"):
        return "hydrogen"
    elif cn.startswith("26") or cn.startswith("72") or cn.startswith("73"):
        return "iron_steel"
    elif cn.startswith("76"):
        return "aluminium"
    elif cn.startswith("28") or cn.startswith("31"):
        return "fertilizers"
    elif cn.startswith("25"):
        return "cement"
    else:
        return None  # electricity and unmapped excluded

df_defaults["material"] = df_defaults["cn_clean"].apply(assign_material)

# Build deduplicated CN code lists per material
materials = ["iron_steel", "aluminium", "cement", "fertilizers", "hydrogen"]
cn_by_material = {}

for material in materials:
    codes = df_defaults[df_defaults["material"] == material]["cn_clean"].unique().tolist()
    cn_by_material[material] = codes
    print(f"{material}: {len(codes)} unique CN codes")

excluded = df_defaults[df_defaults["material"].isna()]["cn_clean"].unique()
print(f"\nExcluded (electricity/unmapped): {len(excluded)} codes")
print(f"Total codes to query: {sum(len(v) for v in cn_by_material.values())}")

iron_steel: 200 unique CN codes
aluminium: 28 unique CN codes
cement: 6 unique CN codes
fertilizers: 27 unique CN codes
hydrogen: 1 unique CN codes

Excluded (electricity/unmapped): 0 codes
Total codes to query: 262


## Section 2: Dataset and Dimension Discovery

Confirmed available COMEXT datasets via `eurostat.get_toc_df(agency='COMEXT')`.
DS-045409 selected as the CN8-level trade dataset. Dimensions and valid
parameter values confirmed below.

In [25]:
DATASET = 'DS-045409'

# Confirm dataset exists and check dimensions
toc = eurostat.get_toc_df(agency='COMEXT')
ds_info = toc[toc['code'] == DATASET][['title', 'code', 'last update of data']]
print("Dataset info:")
print(ds_info.to_string(index=False))

# Confirm dimensions
dims = eurostat.get_pars(DATASET)
print(f"\nDimensions: {dims}")

# Confirm key parameter values
print(f"\nfreq values: {eurostat.get_par_values(DATASET, 'freq')}")
print(f"flow values: {eurostat.get_par_values(DATASET, 'flow')}")
print(f"indicators values: {eurostat.get_par_values(DATASET, 'indicators')}")

reporters = eurostat.get_par_values(DATASET, 'reporter')
eu_codes = [r for r in reporters if 'EU' in str(r)]
print(f"EU reporter codes: {eu_codes}")

Dataset info:
                                 title      code      last update of data
EU trade since 1988 by HS2-4-6 and CN8 DS-045409 2026-04-17T11:00:00+0200

Dimensions: ['freq', 'reporter', 'partner', 'product', 'flow', 'indicators']

freq values: ['A', 'M']
flow values: ['1', '2']
indicators values: ['VALUE_IN_EUROS', 'QUANTITY_IN_100KG', 'SUPPLEMENTARY_QUANTITY']
EU reporter codes: ['EU', 'EU27_2020']


## Section 3: CN Code Format Validation

CBAM defaults contain CN codes at 4-digit (HS4), 6-digit (HS6) and
8-digit (CN8) levels, matching the EU regulation's own classification.

Tested all three lengths against DS-045409 to confirm COMEXT accepts
them as-is. Padded variants (e.g. 72010000, 72024100) are explicitly
rejected by the API with INVALID_QUERY_DIMENSION_VALUE errors.

**Conclusion: CN codes used exactly as stored in cbam_defaults.csv.
No normalization or padding required.**

In [26]:
# Validate CN code formats against API
# Tests: 4-digit, 6-digit, 8-digit originals vs padded equivalents
test_cases = [
    ('7201',     '72010000',  '4-digit HS4'),
    ('720241',   '72024100',  '6-digit HS6'),
    ('72041000', '7204100000','8-digit CN8'),
]

for original, padded, label in test_cases:
    print(f"\n{label}:")
    for code in [original, padded]:
        try:
            df = eurostat.get_data_df(
                DATASET, flags=False,
                filter_pars={
                    'freq': 'A', 'reporter': 'EU27_2020',
                    'partner': 'IN', 'product': code,
                    'flow': '1', 'indicators': 'QUANTITY_IN_100KG'
                }
            )
            status = f"OK - {len(df)} row(s) returned" if df is not None and not df.empty else "empty"
        except Exception as e:
            status = f"REJECTED - {str(e)[:80]}"
        print(f"  {code}: {status}")


4-digit HS4:
  7201: OK - 1 row(s) returned
faultcode: 150
faultstring: INVALID_QUERY_DIMENSION_VALUE: Query is invalid as per its structure's definition. The following values for dimension are not allowed: PRODUCT=72010000.
  72010000: REJECTED - 400 Client Error: Bad Request for url: https://ec.europa.eu/eurostat/api/comext/

6-digit HS6:
  720241: OK - 1 row(s) returned
faultcode: 150
faultstring: INVALID_QUERY_DIMENSION_VALUE: Query is invalid as per its structure's definition. The following values for dimension are not allowed: PRODUCT=72024100.
  72024100: REJECTED - 400 Client Error: Bad Request for url: https://ec.europa.eu/eurostat/api/comext/

8-digit CN8:
  72041000: OK - 1 row(s) returned
faultcode: 150
faultstring: INVALID_QUERY_DIMENSION_VALUE: Query is invalid as per its structure's definition. The following values for dimension are not allowed: PRODUCT=7204100000.
  7204100000: REJECTED - 400 Client Error: Bad Request for url: https://ec.europa.eu/eurostat/api/comext/


## Section 4: Full Extraction

Pulls EU27 import data for all CBAM-covered CN codes by material group.
Both value (euros) and quantity (100kg, converted to tonnes) are extracted.
Years 2020-2024 retained. Partial 2025 data excluded.

Output is long format with one row per country-product-year-indicator.

In [30]:
TARGET_YEARS = [str(y) for y in range(2020, 2025)]
INDICATORS = ['VALUE_IN_EUROS', 'QUANTITY_IN_100KG']

def fetch_material(material, cn_codes):
    """Fetch EU27 import data for a list of CN codes."""
    all_dfs = []
    errors = []

    for i, cn in enumerate(cn_codes):
        try:
            df = eurostat.get_data_df(
                DATASET, flags=False,
                filter_pars={
                    'freq': 'A',
                    'reporter': 'EU27_2020',
                    'product': cn,
                    'flow': '1',
                    'indicators': INDICATORS
                }
            )

            if df is None or df.empty:
                continue

            # Keep only target years
            id_col = 'indicators\\TIME_PERIOD'
            id_vars = ['freq', 'reporter', 'partner', 'product', 'flow', id_col]
            year_cols = [c for c in TARGET_YEARS if c in df.columns]
            df = df[id_vars + year_cols]

            # Melt to long format
            df_long = df.melt(
                id_vars=id_vars,
                var_name='year',
                value_name='value'
            )
            df_long = df_long.rename(columns={id_col: 'indicator'})
            df_long['material'] = material
            df_long = df_long[df_long['value'].notna()]
            all_dfs.append(df_long)

        except Exception as e:
            errors.append((cn, str(e)))

        time.sleep(0.3)

        if (i + 1) % 20 == 0:
            print(f"  {i + 1}/{len(cn_codes)} codes processed...")

    if errors:
        print(f"  Errors on {len(errors)} codes: {errors[:3]}")

    return pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()


# Run extraction by material group
all_results = []

for material in materials:
    codes = cn_by_material[material]
    print(f"\nFetching {material} ({len(codes)} codes)...")
    df_mat = fetch_material(material, codes)
    if not df_mat.empty:
        all_results.append(df_mat)
        print(f"  Done: {len(df_mat)} rows")
    else:
        print(f"  No data returned")

# Combine all materials
df_all = pd.concat(all_results, ignore_index=True)

# Convert quantity from 100kg to tonnes
df_all.loc[df_all['indicator'] == 'QUANTITY_IN_100KG', 'value'] = \
    df_all.loc[df_all['indicator'] == 'QUANTITY_IN_100KG', 'value'] / 10
df_all['indicator'] = df_all['indicator'].replace(
    'QUANTITY_IN_100KG', 'QUANTITY_IN_TONNES'
)

print(f"\nTotal rows extracted: {len(df_all)}")
print(f"Materials: {df_all['material'].unique()}")
print(f"Years: {sorted(df_all['year'].unique())}")
print(f"Partner countries: {df_all['partner'].nunique()}")
print(f"\nSample:")
print(df_all.head(10).to_string())


Fetching iron_steel (200 codes)...
  20/200 codes processed...
  40/200 codes processed...
  60/200 codes processed...
  80/200 codes processed...
  100/200 codes processed...
  120/200 codes processed...
  140/200 codes processed...
  160/200 codes processed...
  180/200 codes processed...
  200/200 codes processed...
  Done: 122820 rows

Fetching aluminium (28 codes)...
  20/28 codes processed...
  Done: 25502 rows

Fetching cement (6 codes)...
  Done: 4108 rows

Fetching fertilizers (27 codes)...
  20/27 codes processed...
  Done: 12286 rows

Fetching hydrogen (1 codes)...
  Done: 466 rows

Total rows extracted: 165182
Materials: <StringArray>
['iron_steel', 'aluminium', 'cement', 'fertilizers', 'hydrogen']
Length: 5, dtype: str
Years: ['2020', '2021', '2022', '2023', '2024']
Partner countries: 244

Sample:
  freq   reporter partner   product flow       indicator  year        value    material
0    A  EU27_2020      AR  26011200    1  VALUE_IN_EUROS  2020         30.0  iron_steel
1

In [31]:
# Save to processed
df_all.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"Final shape: {df_all.shape}")

Saved to: /Users/milcahmaryJoseph/Documents/GitHub/cbam-analysis/data/processed/comext_trade_flows.csv
Final shape: (165182, 9)
